In [ ]:
# @title Setup: Install scpn-quantum-control
!pip install -q scpn-quantum-control
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)


SPDX-License-Identifier: AGPL-3.0-or-later | Commercial license available
Â© Concepts 1996â€“2026 Miroslav Ĺ otek. All rights reserved.
Â© Code 2020â€“2026 Miroslav Ĺ otek. All rights reserved.
ORCID: 0009-0009-3560-0851
Contact: www.anulum.li | protoscience@anulum.li
scpn-quantum-control â€” Aging as Progressive K Decay

If disease = desynchronisation (finding #46), then aging =
progressive coupling decay. As K decreases with age, different
organ systems cross their K_c thresholds at different ages.
The order of failure is predicted by the safety margin.

Uses measured safety margins from morphogenesis notebook (#46)
and measured coupling strengths from bio clocks.

Also: circadian disruption, sarcopenia, neurodegeneration,
cardiac aging â€” all as K(age) crossing K_c.

In [ ]:
import json

In [ ]:
import numpy as np
from scipy import stats

In [ ]:
print("=" * 70)
print("AGING AS PROGRESSIVE K DECAY")
print("=" * 70)

=====================================================================
BIOLOGICAL COUPLING DATABASE (from our findings + literature)
=====================================================================

In [ ]:
# K values normalised to K_c = 1.0 for each system
organ_systems = {
    "circadian_SCN": {
        "K_baseline_ratio": 2.14,  # K/K_c at age 20 (0.3/0.14)
        "decline_rate": 0.015,  # per year (Duffy et al. 2015)
        "K_c": 1.0,
        "failure_phenotype": "sleep fragmentation, phase advance",
        "clinical_onset": 55,  # typical age of noticeable decline
        "mechanism": "VIP receptor loss, SCN neuron death",
    },
    "cortical_gamma": {
        "K_baseline_ratio": 12.0,  # 2.0/0.166
        "decline_rate": 0.010,  # synaptic pruning + myelin loss
        "K_c": 1.0,
        "failure_phenotype": "cognitive decline, reduced working memory",
        "clinical_onset": 65,
        "mechanism": "synaptic density loss, GABAergic decline",
    },
    "cardiac_SA": {
        "K_baseline_ratio": 33.6,  # 5.0/0.149
        "decline_rate": 0.005,  # connexin43 remodelling
        "K_c": 1.0,
        "failure_phenotype": "sick sinus syndrome, atrial fibrillation",
        "clinical_onset": 75,
        "mechanism": "fibrosis, Cx43 redistribution, ion channel remodelling",
    },
    "pancreatic_beta": {
        "K_baseline_ratio": 5.6,  # 1.0/0.179
        "decline_rate": 0.012,  # Cx36 loss + amyloid
        "K_c": 1.0,
        "failure_phenotype": "impaired insulin pulsatility, glucose intolerance",
        "clinical_onset": 50,
        "mechanism": "Cx36 downregulation, islet amyloid",
    },
    "intestinal_ICC": {
        "K_baseline_ratio": 19.1,  # 3.0/0.157
        "decline_rate": 0.008,
        "K_c": 1.0,
        "failure_phenotype": "constipation, dysmotility",
        "clinical_onset": 60,
        "mechanism": "ICC loss, smooth muscle changes",
    },
    "neuromuscular": {
        "K_baseline_ratio": 8.0,  # estimated
        "decline_rate": 0.018,  # fastest decline
        "K_c": 1.0,
        "failure_phenotype": "sarcopenia, motor unit loss",
        "clinical_onset": 40,
        "mechanism": "denervation, NMJ fragmentation, motor neuron death",
    },
    "hippocampal_theta": {
        "K_baseline_ratio": 6.0,  # estimated
        "decline_rate": 0.014,
        "K_c": 1.0,
        "failure_phenotype": "memory decline, spatial navigation loss",
        "clinical_onset": 50,
        "mechanism": "cholinergic loss, theta-gamma decoupling",
    },
    "bone_remodelling": {
        "K_baseline_ratio": 4.0,  # estimated (osteocyte network)
        "decline_rate": 0.020,  # esp. post-menopause
        "K_c": 1.0,
        "failure_phenotype": "osteoporosis, fracture risk",
        "clinical_onset": 50,  # earlier in women
        "mechanism": "osteocyte apoptosis, lacunocanalicular network degradation",
    },
}

In [ ]:
# =====================================================================
# TEST 1: Predict age of failure for each system
# =====================================================================
print("\n" + "=" * 70)
print("TEST 1: PREDICTED AGE OF K_c CROSSING")
print("=" * 70)

In [ ]:
age_ref = 20  # baseline age

In [ ]:
predicted_failures = {}
print(f"{'System':25s} {'K/K_c@20':>8s} {'Rate':>6s} {'Fail@':>7s} {'Clinical':>9s} {'Error':>7s}")
print("-" * 70)

In [ ]:
for name, sys_data in organ_systems.items():
    K_ratio = sys_data["K_baseline_ratio"]
    rate = sys_data["decline_rate"]

    # K(age) = K_baseline * exp(-rate * (age - 20))
    # Failure when K(age) / K_c < 1, i.e. K_baseline_ratio * exp(-rate*(age-20)) = 1
    # age_fail = 20 + ln(K_baseline_ratio) / rate
    if K_ratio > 1:
        age_fail = age_ref + np.log(K_ratio) / rate
    else:
        age_fail = age_ref  # already failing

    clinical = sys_data["clinical_onset"]
    error = age_fail - clinical

    predicted_failures[name] = age_fail
    print(f"{name:25s} {K_ratio:8.1f} {rate:6.3f} {age_fail:7.0f} {clinical:9d} {error:+7.0f}")

In [ ]:
# Correlation between predicted and clinical
pred_ages = np.array([predicted_failures[n] for n in organ_systems])
clin_ages = np.array([organ_systems[n]["clinical_onset"] for n in organ_systems])

In [ ]:
r_age, p_age = stats.pearsonr(pred_ages, clin_ages)
print(f"\nPredicted vs clinical onset: r={r_age:.3f}, p={p_age:.4f}")

In [ ]:
# Mean absolute error
mae = np.mean(np.abs(pred_ages - clin_ages))
print(f"Mean absolute error: {mae:.1f} years")

In [ ]:
# =====================================================================
# TEST 2: K(age) trajectories
# =====================================================================
print("\n" + "=" * 70)
print("TEST 2: K(age) TRAJECTORIES")
print("=" * 70)

In [ ]:
ages = np.arange(20, 100)
print(f"\n{'Age':>4s}", end="")
for name in ["circadian", "cortical", "cardiac", "pancreatic", "neuromusc"]:
    print(f" {name[:10]:>10s}", end="")
print()
print("-" * 60)

In [ ]:
for age in range(20, 100, 10):
    print(f"{age:4d}", end="")
    for _name_full, sys_data in list(organ_systems.items())[:5]:
        K_ratio = sys_data["K_baseline_ratio"] * np.exp(
            -sys_data["decline_rate"] * (age - age_ref)
        )
        marker = "*" if K_ratio < 1.0 else " "
        print(f" {K_ratio:9.2f}{marker}", end="")
    print()

In [ ]:
print("\n* = below K_c (desynchronised)")

In [ ]:
# =====================================================================
# TEST 3: Order of system failure
# =====================================================================
print("\n" + "=" * 70)
print("TEST 3: PREDICTED ORDER OF SYSTEM FAILURE")
print("=" * 70)

In [ ]:
# Sort by predicted failure age
sorted_systems = sorted(predicted_failures.items(), key=lambda x: x[1])
print("\nPredicted cascade of aging:")
for i, (name, age) in enumerate(sorted_systems):
    phenotype = organ_systems[name]["failure_phenotype"]
    mechanism = organ_systems[name]["mechanism"]
    print(f"  {i + 1}. Age ~{age:.0f}: {name}")
    print(f"     Phenotype: {phenotype}")
    print(f"     Mechanism: {mechanism}")

In [ ]:
# =====================================================================
# TEST 4: Interventions as K restoration
# =====================================================================
print("\n" + "=" * 70)
print("TEST 4: INTERVENTIONS AS K RESTORATION")
print("=" * 70)

In [ ]:
interventions = {
    "exercise": {
        "K_boost": 0.3,  # 30% increase in effective K
        "systems": ["cardiac_SA", "neuromuscular", "cortical_gamma"],
        "mechanism": "increased Cx43, BDNF, muscle innervation",
    },
    "caloric_restriction": {
        "K_boost": 0.15,
        "systems": ["circadian_SCN", "pancreatic_beta"],
        "mechanism": "SIRT1 activation, circadian amplitude",
    },
    "bright_light_therapy": {
        "K_boost": 0.20,
        "systems": ["circadian_SCN"],
        "mechanism": "retinohypothalamic input boost",
    },
    "social_interaction": {
        "K_boost": 0.10,
        "systems": ["hippocampal_theta", "cortical_gamma"],
        "mechanism": "theta-gamma coupling from cognitive engagement",
    },
    "gap_junction_drugs": {
        "K_boost": 0.40,
        "systems": ["cardiac_SA", "pancreatic_beta"],
        "mechanism": "connexin upregulation (e.g., rotigaptide)",
    },
}

In [ ]:
print("\nIntervention effects on failure age:")
for intervention, data in interventions.items():
    print(f"\n--- {intervention} (K boost: +{data['K_boost'] * 100:.0f}%) ---")
    print(f"  Mechanism: {data['mechanism']}")
    for sys_name in data["systems"]:
        if sys_name in organ_systems:
            sys_data = organ_systems[sys_name]
            K_new = sys_data["K_baseline_ratio"] * (1 + data["K_boost"])
            rate = sys_data["decline_rate"]
            age_fail_new = age_ref + np.log(K_new) / rate
            age_fail_old = predicted_failures[sys_name]
            gain = age_fail_new - age_fail_old
            print(
                f"  {sys_name:25s}: {age_fail_old:.0f} -> {age_fail_new:.0f} (+{gain:.0f} years)"
            )

In [ ]:
# =====================================================================
# TEST 5: Simulate Kuramoto R vs age for cortical gamma
# =====================================================================
print("\n" + "=" * 70)
print("TEST 5: KURAMOTO R vs AGE (cortical gamma simulation)")
print("=" * 70)

In [ ]:
N_cortical = 30

In [ ]:
def simulate_cortical_aging(K, noise=0.1, dt=0.005, T=200, n_trials=10):
    n_steps = int(T / dt)
    R_trials = []
    for _ in range(n_trials):
        omega = np.random.normal(1.0, 0.15, N_cortical)
        theta = np.random.uniform(0, 2 * np.pi, N_cortical)
        for _s in range(n_steps):
            z = np.mean(np.exp(1j * theta))
            R = abs(z)
            psi = np.angle(z)
            dtheta = omega + K * R * np.sin(psi - theta)
            dtheta += noise * np.random.randn(N_cortical) * np.sqrt(dt)
            theta += dtheta * dt
        z_final = np.mean(np.exp(1j * theta))
        R_trials.append(abs(z_final))
    return np.mean(R_trials)

In [ ]:
sys_cortical = organ_systems["cortical_gamma"]
print(f"{'Age':>4s} {'K/K_c':>8s} {'R_sim':>8s} {'State':>15s}")
print("-" * 40)
for age in range(20, 95, 5):
    K_ratio = sys_cortical["K_baseline_ratio"] * np.exp(
        -sys_cortical["decline_rate"] * (age - age_ref)
    )
    K_actual = K_ratio * 0.166  # actual K value (K_c = 0.166)
    R = simulate_cortical_aging(K_actual, n_trials=8)
    if R > 0.6:
        state = "NORMAL"
    elif R > 0.4:
        state = "mild decline"
    elif R > 0.25:
        state = "MCI"
    else:
        state = "SEVERE"
    print(f"{age:4d} {K_ratio:8.2f} {R:8.3f} {state:>15s}")

In [ ]:
# SYNTHESIS
print("\n" + "=" * 70)
print("SYNTHESIS: AGING AS K DECAY")
print("=" * 70)

In [ ]:
print(f"""
PREDICTIVE MODEL: Age of failure = 20 + ln(K/K_c) / decline_rate

Correlation with clinical onset: r={r_age:.3f}, p={p_age:.4f}
Mean error: {mae:.1f} years

ORDER OF FAILURE (predicted):""")
for i, (name, age) in enumerate(sorted_systems):
    print(f"  {i + 1}. {name:25s} -> age {age:.0f}")

In [ ]:
print("""
KEY INSIGHTS:
1. Safety margin (K/K_c at age 20) determines WHEN a system fails
2. Decline rate determines HOW FAST the margin erodes
3. Interventions that boost K delay failure by ln(1+boost)/rate years
4. Exercise is the most broadly effective (boosts K in 3 systems)
5. The cascade is PREDICTABLE from Kuramoto parameters alone

CLINICAL IMPLICATION:
Measure coupling strength (via EEG coherence, EMG sync, etc.)
to detect K approaching K_c BEFORE clinical symptoms appear.
This is a BIOMARKER of aging at the systems level.
""")

In [ ]:
results = {
    "pred_vs_clinical_r": round(float(r_age), 3),
    "pred_vs_clinical_p": round(float(p_age), 4),
    "mae_years": round(float(mae), 1),
    "failure_order": [name for name, _ in sorted_systems],
    "predicted_ages": {name: round(float(age), 0) for name, age in sorted_systems},
}

In [ ]:
print("\n--- JSON RESULTS ---")
print(json.dumps(results, indent=2))